# Traffic — Load Balancer, App Gateway, Front Door & DNS

Once you have services running and a network they can sit in, the next problem is getting requests to them — fairly, fast, and ideally inspected for malice on the way. Azure has four distinct products that do this, layered from the closest-in to the farthest-out: **Azure Load Balancer**, **Application Gateway**, **Azure Front Door**, and **Traffic Manager**. Each occupies a different layer of the stack, and choosing the right one (or the right combination) is one of the most common architectural questions.

Underneath sits **Azure DNS**, the resolution layer that ties hostnames to all of the above, including the private endpoints that take traffic off the public internet entirely.

## Azure Load Balancer

**Azure Load Balancer** is the layer-4 (TCP/UDP) regional load balancer. It distributes inbound flows across a backend pool based on a hashed five-tuple — no application-layer awareness, just very fast packet steering.

Two SKUs and the choice is permanent:

- **Basic** — free, limited backends, no zone redundancy, no SLA. Being retired in September 2025. Don't start anything new on it.
- **Standard** — production-grade. Zone-redundant or zonal, much higher backend limits, secure-by-default (deny-all unless an explicit rule allows), supports HA Ports, integrates with Standard Public IPs.

Two configurations:

- **Public** — frontend has a public IP. Inbound from the internet to backend VMs/scale sets.
- **Internal (ILB)** — frontend has a private IP in a VNet subnet. Used to load-balance traffic between tiers without exposing them.

Three concepts run the whole thing:

- **Frontend IP configuration** — public or private IPs that listeners attach to.
- **Backend pool** — set of NICs, IPs, or VM scale set instances that receive traffic.
- **Load balancing rules** — frontend port → backend port, with a **health probe** and a session-persistence mode (None, Client IP, Client IP + Protocol).
- **Health probes** — TCP, HTTP, or HTTPS probes that mark backends in or out. Backends that fail the probe stop receiving traffic.

**HA Ports** is a Standard-only mode where the rule covers *all* ports and protocols on the frontend — useful for clustering a network virtual appliance behind a load balancer.

AWS comparison: Azure Load Balancer ≈ Network Load Balancer (NLB). The L4 mental model and the role in an architecture are virtually identical.

## Application Gateway

**Application Gateway** is the regional **layer-7** load balancer — it terminates TLS, inspects HTTP, and routes based on URL path, host header, or query string. It also hosts the **Web Application Firewall (WAF v2)** that protects backends from OWASP Top 10 attacks.

Today only the v2 SKU is current — v1 is retired in April 2026. The features that matter:

- **Listeners** — what to listen on (port + protocol + optional host header). One per public-facing site or path-tree.
- **Rules** — bind a listener to a backend pool with optional path-based routing, URL rewrites, or redirects.
- **Backend pools** — NICs, IPs, FQDNs (you can point at an App Service or a VMSS), or NICs from a private link.
- **Health probes** — HTTP/HTTPS, customisable per pool.
- **TLS termination + end-to-end TLS** — App Gateway holds the public cert; you can re-encrypt to backends with a separate cert (PFX-based).
- **WAF policies** — managed rule sets (Microsoft Default Rule Set, OWASP CRS), bot manager rules, custom rules. Attach a policy to a listener or to specific paths.
- **Autoscale** — set min/max instance count; v2 scales automatically with traffic.
- **Zone redundancy** — spread instances across availability zones; the default for production deployments.

**Application Gateway Ingress Controller (AGIC)** makes the gateway act as a Kubernetes ingress for an AKS cluster — declared with Ingress and IngressBackend CRDs.

AWS comparison: Application Gateway ≈ Application Load Balancer + AWS WAF combined. Single product on Azure side; two services on AWS.

## Azure Front Door

**Azure Front Door** is the global edge — anycast IPs in over 100 Microsoft POPs worldwide that terminate TLS, run a WAF, cache static content, and route requests to your nearest healthy origin.

The model is similar to App Gateway but operates globally rather than regionally:

- **Origins** — your actual backends (App Services, Storage, App Gateways, public IPs). Grouped into **origin groups** with a load-balancing config.
- **Endpoints** — the customer-facing host (`myapp-xyz123.z01.azurefd.net` or your own domain).
- **Routes** — match patterns (path + host) and forward to an origin group, with optional caching and **rules-engine** transformations.
- **WAF policy** — same OWASP CRS rule sets as App Gateway, but enforced at the edge before requests reach your region.
- **Caching** — TTL-aware cache at every POP. Compression, query-string handling, vary-by headers.
- **Private Link integration** — Premium tier can reach private origins through Private Link, so the origin itself can have no public IP at all.

Two SKUs: **Standard** (basic CDN + WAF) and **Premium** (managed WAF rules, bot protection, Private Link to origins). Premium is the production default for anything internet-facing.

AWS comparison: Front Door ≈ CloudFront + AWS WAF + Global Accelerator merged into one product. The anycast IPs and the edge POP topology look like CloudFront; the rule engine and WAF feel like the AWS WAF + CloudFront combination wired together for you.

## Traffic Manager

**Traffic Manager** is **DNS-based** global routing — it doesn't carry traffic, it resolves your hostname to one of several endpoint IPs based on a routing method. The client then connects directly to the resolved IP.

Routing methods:

- **Priority** — primary endpoint preferred; failover to secondary if primary is unhealthy. Classic active-passive.
- **Weighted** — distribute by weight (60/40, etc.).
- **Performance** — return the endpoint with the lowest network latency to the resolver.
- **Geographic** — return based on the resolver's country/region.
- **Multi-value** — return up to N healthy endpoints.
- **Subnet** — match the resolver's subnet to a configured endpoint.

Because Traffic Manager works at DNS, the failover speed is bounded by TTL — typically 60 seconds before clients learn about a swap. That's slow compared to Front Door's anycast-driven instant rerouting. Traffic Manager is the right choice for **non-HTTP** workloads (anything beyond HTTP/HTTPS), or for endpoints that are not Azure-resident (you can point at AWS or on-prem IPs).

AWS comparison: Traffic Manager ≈ Route 53 routing policies. The mental model is the same — DNS as the routing primitive.

## Azure CDN — and why it's mostly gone

Historically, **Azure CDN** was a separate product with three providers (Microsoft, Verizon, Akamai). Verizon and Akamai CDN profiles are retired; the remaining Microsoft profile is being superseded by **Azure Front Door Standard / Premium**, which absorbed the CDN feature set.

If you see Azure CDN in older docs or in an existing tenant, treat it as a deprecated path — new deployments should use Front Door, which gives you CDN, WAF, anycast routing, and rules engine in one product.

AWS comparison: There is no longer a standalone CDN on Azure to compare to CloudFront; Front Door is the CDN now.

## Choosing the right traffic service

The four products overlap enough to confuse, and they compose enough to coexist. The decision tree:

```
Is traffic HTTP/HTTPS?
 ├── No → Traffic Manager (DNS) or Azure Load Balancer (within VNet)
 └── Yes:
      Is it global and internet-facing?
       ├── Yes → Azure Front Door (edge, WAF, caching)
       └── No (regional / internal):
            Need WAF, path-routing, TLS termination?
             ├── Yes → Application Gateway (regional L7)
             └── No  → Azure Load Balancer (L4) or App Service built-in
```

The most common production stack for an internet-facing app: **Front Door** at the edge, **Application Gateway** (or App Service) regionally, **internal Load Balancer** between tiers. Front Door handles the WAF, caching, and global routing; App Gateway handles regional path routing and second-line WAF; the internal LB load-balances private traffic between tiers. The three layers don't fight — they each solve a different problem at a different scope.

Note: **Azure Front Door + Application Gateway** with WAF enabled on *both* is a common pattern. Front Door catches the broad volumetric/OWASP attacks at the edge; App Gateway WAF runs region-level rules that can include backend-specific logic.

## Azure DNS

**Azure DNS** hosts authoritative DNS zones — both **public** (delegated from `.com`, `.io`, etc.) and **private** (resolvable only inside linked VNets).

Three record types worth knowing beyond the obvious A/AAAA/CNAME/TXT:

- **Alias records** — special A/AAAA records that point directly at an Azure resource (Public IP, Front Door, Traffic Manager, App Service, CDN). The DNS answer follows the underlying resource's IP automatically — no TTL gotchas when the resource scales or moves.
- **Apex records** — the zone apex (`contoso.com`) cannot hold a CNAME by RFC. Alias records solve this — you can point `contoso.com` at a Traffic Manager or Front Door without manual A-record management.
- **Auto-registration** — for private zones, VNet-attached resources can self-register their hostnames, so VMs in `vnet-app` resolve each other by name without anyone editing the zone.

**Private DNS zones** are the unsung hero of private-endpoint deployments — every Azure PaaS that supports Private Link has a corresponding `privatelink.<service>.core.windows.net` zone you link to your VNets so clients resolve to the private IP.

**Azure Private DNS Resolver** is a managed forwarder you put in the hub VNet for cross-VNet, hybrid, or split-horizon resolution. Before it existed, the standard pattern was a Linux VM running BIND in the hub — Private DNS Resolver replaces that VM with a managed service.

In [ ]:
# Stand up a regional Application Gateway in front of a Linux VMSS.

RG=rg-traffic-demo
LOC=eastus
az group create -n $RG -l $LOC

az network vnet create -g $RG -n vnet-app \
  --address-prefix 10.30.0.0/16 \
  --subnet-name snet-agw --subnet-prefix 10.30.0.0/24
az network vnet subnet create -g $RG --vnet-name vnet-app \
  --name snet-backend --address-prefix 10.30.1.0/24

# 1. Public IP for the gateway (Standard SKU, zone redundant).
az network public-ip create -g $RG -n pip-agw \
  --sku Standard --allocation-method Static --zone 1 2 3

# 2. Application Gateway v2, WAF v2, autoscale.
az network application-gateway create -g $RG -n agw-app \
  --location $LOC --sku WAF_v2 \
  --public-ip-address pip-agw \
  --vnet-name vnet-app --subnet snet-agw \
  --min-capacity 2 --max-capacity 10 --zones 1 2 3 \
  --priority 100

# 3. Add a WAF policy in Prevention mode with Microsoft Default Rule Set.
az network application-gateway waf-policy create -g $RG -n waf-prod \
  --type OWASP --version 3.2
az network application-gateway waf-policy policy-setting update \
  -g $RG --policy-name waf-prod --mode Prevention --state Enabled

# 4. Front Door Premium in front of the App Gateway (multi-region pattern).
az afd profile create -g $RG --profile-name fd-prod --sku Premium_AzureFrontDoor
az afd endpoint create -g $RG --profile-name fd-prod --endpoint-name app-prod
# (origin groups, routes, custom domain are configured next.)

## Putting it together

A typical internet-facing production stack, end to end:

1. **Custom domain** (`app.contoso.com`) at the apex, registered with your registrar, delegated to **Azure DNS**.
2. An **Alias record** in Azure DNS pointing the apex at the **Azure Front Door** endpoint.
3. **Front Door Premium** handles TLS, WAF (Microsoft Default Rule Set), caching, and origin selection. Routes traffic to the **App Gateway** in the primary region; an origin group fails over to a secondary-region App Gateway if the primary is unhealthy.
4. **Application Gateway WAF v2** in each region terminates the internal TLS, runs region-level WAF and path-based routing, sends requests into the backend pool.
5. The backend pool points at an **App Service** (or AKS via AGIC, or a VMSS), with **internal Load Balancers** between any private tiers (cache, database).
6. **Private DNS zones** linked to the VNets resolve private endpoints for Storage, SQL, Key Vault, and anything else the workload depends on.

That shape covers the four traffic products plus DNS and answers the design question in one go. Smaller workloads peel layers off as needed — a single-region app might run with App Gateway + App Service and no Front Door; an internal-only API might be just an internal Load Balancer or a private App Service. But the layered model is what scales when the requirements scale.